# Poster results — reproducing every number and chart on the poster

One notebook, read top to bottom, regenerates every figure and every headline number that appears
on `poster/poster.html`, plus two checks that were added after the poster's first draft (§6/§9
below) and had never been formalised anywhere. It reads already-saved artifacts wherever they
exist — nothing here re-fits a model or re-runs conformal calibration — so it is cheap (~1 min)
and needs no GPU.

Sections match the poster's own card numbers.

## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")   # notebooks/ is one level down, so the repo root goes on the path

import warnings
warnings.filterwarnings("ignore")

import json
import shutil

import numpy as np
import pandas as pd

from src import orders, recovery
from src.utils import config, plots
from src.utils.features import censoring_bucket
from src.utils.metrics import lost_sales_vs_waste, quantile_scores, pinball_by_quantile, scores_by_bucket

REPORTS = config.REPORTS_DIR
PLOTS = config.PLOTS_DIR
POSTER_IMAGES = config.ROOT / "poster" / "images"


## 2. Data — the censoring rate, and when a stockout first begins

The 43.8% headline is read straight off the subset summary. The hourly EDA is new: it answers
"once a shelf empties, does it tend to come back the same day?" — training-period days only,
matching every other recovery figure.

In [2]:
daily = recovery.load_daily("recovered")
hourly = recovery.hours(daily)
lo, hi = config.ACTIVE_HOURS

tr = hourly[hourly.period == "training"]
active = tr[(tr.hour >= lo) & (tr.hour < hi)]

out_hours = (active[active.hour_stockout == 1]
             .groupby(["store_id", "product_id", "dt"])["hour"].apply(list))
first_hour = out_hours.apply(min)
recovers_same_day = out_hours.apply(
    lambda hrs: any(h > min(hrs) and h not in hrs for h in range(min(hrs), hi)))

hours = np.arange(lo, hi)
pct = first_hour.value_counts().reindex(hours, fill_value=0) / len(first_hour) * 100

timing = pd.DataFrame({"hour": hours, "pct_of_stockout_days_starting_here": pct.values})
timing.to_csv(REPORTS / "stockout_timing.csv", index=False)

already_open_pct = float(pct.values[0])
never_recovers_pct = float(100 - recovers_same_day.mean() * 100)
avg_hours_out = float(out_hours.apply(len).mean())

print(f"n stockout-days (training period): {len(out_hours):,}")
print(f"already empty at opening (06:00): {already_open_pct:.1f}%")
print(f"never restock before closing, once started: {never_recovers_pct:.1f}%")
print(f"avg hours affected per stockout day: {avg_hours_out:.2f} of {hi - lo}")


loading recovered subset from data/processed


loading hourly subset from data/processed


n stockout-days (training period): 156,969
already empty at opening (06:00): 13.4%
never restock before closing, once started: 87.4%
avg hours affected per stockout day: 7.35 of 16


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 14

fig, ax = plt.subplots(figsize=(7.6, 7.1), dpi=300)
colors = ["#2a78d6"] + ["#1baf7a"] * (len(hours) - 1)
ax.bar(hours, pct.values, color=colors, width=0.82, zorder=3)
ax.set_ylim(0, 19)
ax.annotate(f"already empty\nat opening ({already_open_pct:.1f}%)",
            xy=(hours[0], pct.values[0]), xytext=(hours[0] + 3.0, pct.values[0] + 2.6),
            fontsize=13, fontweight="bold", color="#0b0b0b", ha="left",
            arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#0b0b0b"))
ax.text(0.97, 0.97, f"{never_recovers_pct:.1f}% of stockouts, once\nstarted, never restock\nbefore closing",
        transform=ax.transAxes, ha="right", va="top", fontsize=13, color="#2f5d56",
        bbox=dict(boxstyle="round,pad=0.5", fc="#1baf7a1f", ec="#2f5d56", lw=1.0))
ax.set_xticks(hours[::3])
ax.set_xticklabels([f"{h:02d}:00" for h in hours[::3]], fontsize=13)
ax.set_xlabel("hour the shelf first goes empty", fontsize=13.5, color="#52514e")
ax.set_ylabel("share of stockout-days (%)", fontsize=13.5, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_title("When stockouts first begin", fontsize=18, pad=20, loc="left")
fig.tight_layout()
fig.savefig(PLOTS / "stockout_timing.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "stockout_timing.png", POSTER_IMAGES / "stockout_timing.png")
plt.close(fig)
print("saved outputs/plots/stockout_timing.png (+ copied to poster/images/)")


saved outputs/plots/stockout_timing.png (+ copied to poster/images/)


## 4. Stage-wise results

Every table below is computed here from the saved artifacts — the recovery model board, the
four validation forecast parquets, the conformal reports — not transcribed from anywhere else.
If a number upstream changes, these change with it.

### Recovery — validated on held-out full-shelf days

In [4]:
ARMS = [(family, target) for family in ("tft", "xgb") for target in ("recovered", "raw")]

rec = pd.read_csv(config.RECOVERY_COMPARISON, index_col=0)
params, leak = (json.loads(config.RECOVERY_PARAMS.read_text()),
                json.loads(config.LEAKAGE_CHECKS.read_text()))
chosen, control = params["model"], "series_hour_mean"
recovery_tbl = pd.DataFrame([
    ("chosen Stage-1 model", chosen),
    ("day WAPE on held-out full-shelf days", f"{rec.loc[chosen, 'wape']:.4f}"),
    (f"no-model control ({control})", f"{rec.loc[control, 'wape']:.4f}"),
    ("better than the control by", f"{100 * (1 - rec.loc[chosen, 'wape'] / rec.loc[control, 'wape']):.1f}%"),
    ("aggregation bias before correction", f"{100 * rec.loc[chosen, 'wpe_uncorrected']:+.1f}%"),
    ("one fitted correction multiplier", f"x{rec.loc[chosen, 'bias_correction']:.4f}"),
    ("day WPE after correction", f"{rec.loc[chosen, 'wpe']:+.4f}"),
    ("leakage checks passed", f"{sum(leak.values())}/{len(leak)}"),
], columns=["what", "value"])
recovery_tbl.to_csv(REPORTS / "conclusion_recovery.csv")
print("A. DID RECOVERY WORK?  scored where recorded sales ARE demand, so it could have failed")
print("   source: reports/recovery_model_comparison.csv + recovery_params.json + leakage_checks.json")
print(recovery_tbl.to_string(index=False))

A. DID RECOVERY WORK?  scored where recorded sales ARE demand, so it could have failed
   source: reports/recovery_model_comparison.csv + recovery_params.json + leakage_checks.json
                                what            value
                chosen Stage-1 model lightgbm_tweedie
day WAPE on held-out full-shelf days           0.2966
 no-model control (series_hour_mean)           0.3684
          better than the control by            19.5%
  aggregation bias before correction           +13.2%
    one fitted correction multiplier          x0.9001
            day WPE after correction          +0.0186
               leakage checks passed              5/5


### Recovery, split by how often a series sells out — two model families, same answer

In [5]:
by_band = pd.read_csv(REPORTS / "recovery_by_censoring_bucket.csv", index_col=0)
print(by_band.to_string(index=False))


   days  recorded_mean  recovered_mean  uplift_%  uplift_%_flat_fill  model_vs_flat_%
71589.0           1.41            1.52      7.98               14.75            54.10
83405.0           1.16            1.49     28.75               42.10            68.29
48999.0           0.83            1.40     69.52               99.31            70.00
32587.0           0.21            1.17    445.84              556.23            80.15


### The same split, on the live forecasts — does it change the forecast, not just Stage-1's own validation?

The table above is Stage-1's own validation (recovered vs. a no-model control, on days recorded
sales are known). This is a different question: does training the **forecaster** on recovered
demand instead of raw sales change its accuracy, and does the improvement concentrate in the same
chronic-stockout band, in **both** model families? Computed from the four validation forecast
parquets.

In [6]:
val_fc = {f"{f}_{t}": pd.read_parquet(config.forecast_parquet("validation", t, f)) for f, t in ARMS}
band = censoring_bucket(daily)          # each series' share of TRAINING days that sold out

cols = {}
for fam in ("tft", "xgb"):
    t = pd.concat({tag: scores_by_bucket(val_fc[f"{fam}_{tag}"], band) for tag in ("raw", "recovered")},
                  axis=1)
    cols["n_scored"] = t[("raw", "n_scored")].astype(int)
    cols[f"{fam} WAPE change %"] = ((t[("recovered", "WAPE")] / t[("raw", "WAPE")] - 1) * 100).round(2)
    cols[f"{fam} WPE raw"] = t[("raw", "WPE")].round(3)
    cols[f"{fam} WPE recovered"] = t[("recovered", "WPE")].round(3)
band_tbl = pd.DataFrame(cols)
band_tbl.to_csv(REPORTS / "conclusion_by_band.csv")
print("C. DOES RECOVERY CHANGE THE FORECAST?  negative WAPE change = recovery is MORE accurate")
print("   computed live from the four validation forecast parquets, split by censoring band")
print(band_tbl.to_string())

C. DOES RECOVERY CHANGE THE FORECAST?  negative WAPE change = recovery is MORE accurate
   computed live from the four validation forecast parquets, split by censoring band
               n_scored  tft WAPE change %  tft WPE raw  tft WPE recovered  xgb WAPE change %  xgb WPE raw  xgb WPE recovered
<25% censored      3955              -0.57       -0.030              0.071              -1.45        0.020              0.118
25-50%            30345               3.16       -0.044              0.105               2.85        0.004              0.112
50-75%            12319               0.23       -0.067              0.084               1.23       -0.017              0.111
>=75%              1375             -25.24       -0.202             -0.003             -22.91       -0.205              0.001
ALL               47994              -0.06       -0.066              0.085               0.03       -0.022              0.100


### Are the bands honest?

Read directly off the saved conformal results for all four arms, both windows — nothing here is
recomputed, this just assembles what `conformal.run` already wrote to disk. Alongside the raw
coverage numbers: a day-block bootstrap 95% CI on coverage, and a `nominal_in_ci` verdict column
— `True` means the band's coverage is statistically indistinguishable from what it claims,
`False` means the miss is real.

In [7]:
# ---------------------------------------------------------------- D. are the bands honest?
# Read straight out of the conformal report JSONs, so this table cannot drift from what was fitted.
# `nominal_in_ci` is the verdict column: True = the band's coverage is statistically
# indistinguishable from what it claims; False = the miss is real, and the sign says which way.
BANDS = [(0.80, "wide80"), (0.95, "wide95")]

rows = []
for family, target in ARMS:
    for nominal, tag in BANDS:
        path = config.conformal_results(tag, family=family, target=target)
        if not path.exists():
            continue
        for window, d in json.loads(path.read_text())["periods"].items():
            ci = d["coverage_ci"]
            rows.append({"arm": f"{family}_{target}", "band": f"{nominal:.0%}", "window": window,
                         "claims": nominal, "before": d["uncorrected"]["coverage"],
                         "after": d["corrected"]["coverage"],
                         "ci_low": ci["ci_low"], "ci_high": ci["ci_high"],
                         "offset": d["offset"], "drift": d["drift_inflation"],
                         "nominal_in_ci": bool(ci["ci_low"] <= nominal <= ci["ci_high"])})

calibration_tbl = (pd.DataFrame(rows)
                   .sort_values(["window", "band", "arm"], ascending=[False, True, True])
                   .reset_index(drop=True))
print("D. ARE THE BANDS HONEST?  does an 80% band contain the truth 80% of the time?")
print("   source: reports/conformal_results_<family>_<target>_<band>.json")
print(calibration_tbl.to_string(index=False))

n_pass = calibration_tbl.nominal_in_ci.sum()
print(f"\n   {n_pass} of {len(calibration_tbl)} (arm x band x window) combinations land statistically")
print("   on their nominal level. Note the pattern rather than the count:")
for w, g in calibration_tbl.groupby("window", sort=False):
    print(f"     {w:<11} raw coverage {g.before.min():.3f}-{g.before.max():.3f} -> "
          f"corrected {g.after.min():.3f}-{g.after.max():.3f}   "
          f"{g.nominal_in_ci.sum()}/{len(g)} contain nominal")
print("   validation OVERSHOOTS (bands slightly too wide); test UNDERSHOOTS on the 80% band for the")
print("   tree arms. One offset fitted at one distance in time cannot serve both distances at once.")

calibration_tbl.to_csv(REPORTS / "conclusion_calibration.csv", index=False)

D. ARE THE BANDS HONEST?  does an 80% band contain the truth 80% of the time?
   source: reports/conformal_results_<family>_<target>_<band>.json
          arm band     window  claims  before  after  ci_low  ci_high  offset  drift  nominal_in_ci
      tft_raw  80% validation    0.80  0.7494 0.8727  0.8579   0.8860  0.1216   0.00          False
tft_recovered  80% validation    0.80  0.7378 0.8255  0.8128   0.8376  0.0775   0.00          False
      xgb_raw  80% validation    0.80  0.7758 0.8584  0.8451   0.8710  0.0803   0.00          False
xgb_recovered  80% validation    0.80  0.7395 0.8455  0.8280   0.8612  0.0965   0.00          False
      tft_raw  95% validation    0.95  0.9215 0.9826  0.9767   0.9875  0.3140   0.00          False
tft_recovered  95% validation    0.95  0.9179 0.9702  0.9645   0.9753  0.1704   0.00          False
      xgb_raw  95% validation    0.95  0.9440 0.9774  0.9718   0.9819  0.1473   0.00          False
xgb_recovered  95% validation    0.95  0.9193 0.9703  0

### At matched 95% demand met — the waste_comparison chart

Every arm held to the same 95% demand-met target, so waste differences reflect forecast quality
rather than one arm simply ordering more (E1). The deciding comparison is raw against its own
recovered twin at that identical availability (E2) — the claim the project stands on. A third
view holds the cost ratio fixed instead (`q*=0.80`) and compares every arm against the naive rule
on both windows (E3).

In [8]:
# Built fresh here via orders.load_forecast rather than inherited from notebook 04's kernel -
# every arm that has a validation forecast, plus whichever also has a test forecast on disk
# (notebook 04 section 4 is what puts test forecasts there; this only checks what exists).
frames = {f"{f}_{t}": orders.load_forecast(period="validation", family=f, target=t) for f, t in ARMS}
test_frames = {}
for f, t in ARMS:
    try:
        test_frames[f"{f}_{t}"] = orders.load_forecast(period="test", family=f, target=t)
    except FileNotFoundError:
        pass
windows = {"validation": frames, "test": test_frames}

# E1 - every arm held to the SAME availability, so waste is what forecast quality bought and not
# a side effect of one arm simply ordering more.
met = pd.concat([orders.at_demand_met(fr, 0.95, verbose=False).assign(window=w)
                 for w, fr in windows.items() if fr], ignore_index=True)
met = met[["window", "arm", "order_percentile", "waste_pct", "stockout_pct"]]
print("E1. EVERY ARM HELD TO 95% OF DEMAND MET  -  what does each one waste to get there?")
print("    full-shelf days only: recorded sales ARE demand there, and those are the quiet days where")
print("    a recovered model over-orders, so this regime PENALISES recovery")
print(met.to_string(index=False))

# E2 - the deciding comparison: raw against its own recovered twin, same family, same availability.
# NOT waste_at_equal_service.csv - notebook 04 no longer writes that file; it only ever held
# whichever single validation-only period was last computed there. This is the two-window table
# the poster's Card 4 waste_comparison chart actually reads, built below once both windows are open.
pairs = []
for w, g in met.groupby("window", sort=False):
    s = g.set_index("arm")
    for fam, nice in [("tft", "TFT"), ("xgb", "XGBoost")]:
        if not {f"{fam}_raw", f"{fam}_recovered"} <= set(s.index):
            continue
        r, c = s.loc[f"{fam}_raw"], s.loc[f"{fam}_recovered"]
        pairs.append({"window": w, "model": nice,
                      "raw waste %": r.waste_pct, "recovered waste %": c.waste_pct,
                      "waste saved (pts)": round(c.waste_pct - r.waste_pct, 1),
                      "raw orders at": f"q{r.order_percentile:.2f}",
                      "recovered orders at": f"q{c.order_percentile:.2f}"})
raw_vs_rec = pd.DataFrame(pairs)
print("\n\nE2. RAW vs RECOVERED AT IDENTICAL AVAILABILITY  -  the claim the project stands on")
print(raw_vs_rec.to_string(index=False))
missing = [f"{f}_{t}" for f, t in ARMS if f"{f}_{t}" not in test_frames]
if missing:
    print(f"    no test forecast for: {', '.join(missing)} - those pairs are validation-only")

# E3 - the headline operating point against the status quo, every arm, both windows.
head = []
for w, fr in windows.items():
    for name, (df, qcols) in fr.items():
        k = orders.run(df, qcols, period=w, regime="observed", save=False, verbose=False)["kpi"]
        head.append({"window": w, "arm": name, "n_scored": k["n_scored"], **k["headline"]})
headline_tbl = pd.DataFrame(head).drop(columns=["c_u", "c_o"])
print("\n\nE3. HEADLINE  q*=0.80 (a stockout assumed to cost 4x a bin) against the naive rule")
print("    positive cost_vs_naive_pct = CHEAPER than the naive rule")
print(headline_tbl.to_string(index=False))

for name, t in [("conclusion_demand_met", met), ("conclusion_raw_vs_recovered", raw_vs_rec),
                ("conclusion_headline", headline_tbl)]:
    t.to_csv(REPORTS / f"{name}.csv", index=False)

E1. EVERY ARM HELD TO 95% OF DEMAND MET  -  what does each one waste to get there?
    full-shelf days only: recorded sales ARE demand there, and those are the quiet days where
    a recovered model over-orders, so this regime PENALISES recovery
    window           arm  order_percentile  waste_pct  stockout_pct
validation tft_recovered             0.696       41.4         15.55
validation xgb_recovered             0.679       43.2         14.66
validation       tft_raw             0.795       44.0         14.93
validation       xgb_raw             0.793       48.5         13.06
      test tft_recovered             0.920       51.9         10.54
      test       tft_raw             0.947       72.7          7.09
      test xgb_recovered             0.937       75.8          7.13
      test       xgb_raw             0.968       95.9          5.10


E2. RAW vs RECOVERED AT IDENTICAL AVAILABILITY  -  the claim the project stands on
    window   model  raw waste %  recovered waste %  waste



E3. HEADLINE  q*=0.80 (a stockout assumed to cost 4x a bin) against the naive rule
    positive cost_vs_naive_pct = CHEAPER than the naive rule
    window           arm  n_scored  q_star  waste_vs_naive_pct  cost_vs_naive_pct  stockout_pct  demand_met_pct
validation tft_recovered     47994     0.8             -148.03              34.72          9.27           96.88
validation       tft_raw     47994     0.8             -104.00              37.26         14.56           95.12
validation xgb_recovered     47994     0.8             -173.52              30.68          7.74           97.24
validation       xgb_raw     47994     0.8             -125.67              32.76         12.58           95.16
      test tft_recovered     23402     0.8              -63.40              27.97         22.64           90.19
      test       tft_raw     23402     0.8              -39.41               9.22         29.23           84.55
      test xgb_recovered     23402     0.8             -117.14        

The poster's `waste_comparison.png` is a custom-styled version of this same table (grouped
bars, one panel per window, the point-improvement annotated directly) — reproduced here from the
same source file rather than via `plots.plot_waste_comparison` (which draws the plainer,
notebook-native version of the same numbers, kept in `outputs/plots/` for that purpose and left
untouched).

In [9]:
windows = list(raw_vs_rec["window"].unique())
fig, axes = plt.subplots(1, len(windows), figsize=(6.5 * len(windows), 4.6), dpi=300, squeeze=False)
for ax, window in zip(axes[0], windows):
    sub = raw_vs_rec[raw_vs_rec["window"] == window].reset_index(drop=True)
    x = np.arange(len(sub))
    width = 0.32
    b1 = ax.bar(x - width / 2, sub["raw waste %"], width, color="#2a78d6", label="trained on raw sales")
    b2 = ax.bar(x + width / 2, sub["recovered waste %"], width, color="#eb6834", label="trained on recovered demand")
    for bars in (b1, b2):
        for b in bars:
            ax.annotate(f"{b.get_height():.1f}%", (b.get_x() + b.get_width() / 2, b.get_height()),
                       xytext=(0, 3), textcoords="offset points", ha="center", fontsize=11)
    for xi, row in sub.iterrows():
        top = max(row["raw waste %"], row["recovered waste %"])
        ax.annotate(f"{row['waste saved (pts)']:+.1f} pts", (xi, top + top * 0.12), ha="center",
                   fontsize=12, fontweight="bold", color="#0ca30c")
    ax.set_xticks(list(x))
    ax.set_xticklabels(sub["model"], fontsize=13)
    ax.set_ylabel("food wasted\n(% of demand)", fontsize=12.5, color="#52514e")
    ax.set_title(f"{window.capitalize()}, at 95% demand met", fontsize=14)
    ax.set_ylim(0, sub[["raw waste %", "recovered waste %"]].to_numpy().max() * 1.32)
    ax.yaxis.grid(True, color="#e1e0d9", linewidth=1)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.08), frameon=False, fontsize=12)
fig.suptitle("Recovered wastes less than raw, at identical availability", y=1.18, fontsize=15)
fig.tight_layout()
fig.savefig(PLOTS / "waste_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "waste_comparison.png", POSTER_IMAGES / "waste_comparison.png")
plt.close(fig)
print("saved outputs/plots/waste_comparison.png (+ copied to poster/images/)")


saved outputs/plots/waste_comparison.png (+ copied to poster/images/)


## 5. Final ordering decision — the cost sweep

Validation window, recovered TFT arm — `reports/cost_sweep.csv`. The sealed test week is scored
only at the single headline point (§4 above and the four-arm table in §6).

In [10]:
sweep = pd.read_csv(REPORTS / "cost_sweep.csv")
print(sweep.to_string(index=False))
print()
print(f"cheaper than naive at all {len(sweep)} ratios tested, "
      f"{sweep.cost_vs_naive_pct.min():.1f}% to {sweep.cost_vs_naive_pct.max():.1f}%")


  c_u  c_o  q_star  waste_model  waste_naive  waste_vs_naive_pct  stockout_pct_model  stockout_pct_naive  demand_met_pct_model  demand_met_pct_naive  cost_model  cost_naive  cost_vs_naive_pct
 0.25  1.0  0.2000      2186.79     10792.39               79.74               77.47                41.7                 66.70                 79.91     6292.39    13269.74              52.58
 0.50  1.0  0.3333      4830.16     10792.39               55.24               59.53                41.7                 77.95                 79.91    10266.17    15747.08              34.81
 1.00  1.0  0.5000     10186.52     10792.39                5.61               36.54                41.7                 87.86                 79.91    16172.70    20701.77              21.88
 2.00  1.0  0.6667     18703.70     10792.39              -73.30               17.93                41.7                 94.28                 79.91    24343.60    30611.16              20.47
 3.00  1.0  0.7500     23651.35     1079

In [11]:
model_pct_of_naive = (100 - sweep["cost_vs_naive_pct"]).round(1)

fig, ax = plt.subplots(figsize=(9.8, 5.8), dpi=300)
x = range(len(sweep))
ax.axhline(100, color="#898781", linewidth=2.2, linestyle=(0, (5, 3)), zorder=2,
           label="naive: order what sold last week")
ax.plot(x, model_pct_of_naive, marker="o", markersize=8, linewidth=2.6, color="#eb6834",
        zorder=3, label="model: newsvendor order")
for i, v in enumerate(model_pct_of_naive):
    if i in (0, 3, len(sweep) - 1):
        ax.annotate(f"{v:.0f}%", (i, v), textcoords="offset points", xytext=(0, -22),
                    ha="center", fontsize=14, fontweight="bold", color="#0b0b0b")
ax.set_xticks(list(x))
ax.set_xticklabels([f"{v:g}\u00d7" for v in sweep["c_u"]], fontsize=13)
ax.set_xlabel("assumed cost ratio  (stockout cost \u00f7 waste cost)", fontsize=13.5, color="#52514e")
ax.set_ylabel("ordering cost, indexed to naive = 100%", fontsize=13.5, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
ax.set_ylim(0, 118)
for s in ("top", "right", "bottom"):
    ax.spines[s].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_title("The model is cheaper than naive at every cost ratio", fontsize=18, pad=14, loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False, fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS / "cost_sweep.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "cost_sweep.png", POSTER_IMAGES / "cost_sweep.png")
plt.close(fig)
print("saved outputs/plots/cost_sweep.png (+ copied to poster/images/)")


saved outputs/plots/cost_sweep.png (+ copied to poster/images/)


## 6. Choosing an operating point

Validation window, full-shelf days, recovered TFT arm. Computed below rather than read back,
since the strict-dominance window it finds is worth showing the search for, not just the result.

In [12]:
# ------------------------------------------- E5. which quantile, and for whom?
# EVERY ROW HERE IS SELF-CONSISTENT. `q*` and `c_u/c_o` are one number written two ways
# (q* = c_u/(c_u+c_o), so with c_o fixed at 1 the ratio is exactly q*/(1-q*)). Each row therefore
# reads "IF you believe a stockout costs this much, order here, and these are the outcomes."
# The ratio is NOT held fixed down the table - it varies with q*, by definition. The next
# subsection does the opposite, and says so there.
#
# Neither q* nor the ratio means anything to a shopkeeper, so both are mapped onto outcomes a shop
# feels, and compared against TODAY - the one reference point needing no cost assumption at all.
model_df, model_qcols = orders.load_forecast(period="validation", family="tft", target="recovered")
demand, mask = orders.realised_demand(model_df, "observed")
naive_order = orders.naive_orders(model_df)[mask]
total = demand.sum()

def outcome(q_star):
    """(waste units, % of demand met, % of product-days that ran out) at one order percentile."""
    o = orders.order_quantity(model_df, q_star, model_qcols)[mask]
    return (float(np.maximum(o - demand, 0).sum()),
            100 * float(1 - np.maximum(demand - o, 0).sum() / total),
            100 * float((demand > o).mean()))

naive_waste = float(np.maximum(naive_order - demand, 0).sum())
naive_met = 100 * float(1 - np.maximum(demand - naive_order, 0).sum() / total)
naive_out = 100 * float((demand > naive_order).mean())
print(f"TODAY (the naive rule): waste {naive_waste:,.0f} units | demand met {naive_met:.1f}% | "
      f"ran out on {naive_out:.1f}% of product-days")

# Three boundaries against today, each ONE-SIDED and reported separately. No waste ceiling is
# invented: a tolerable waste level is a business constraint nobody supplied, and inventing one is
# exactly what the cost sweep exists to avoid. An earlier version of this cell collapsed these into a
# single "viable" flag, which passed a policy wasting 4x today as fine. Three columns, no verdict.
grid = np.arange(0.30, 0.99, 0.001)
stats = np.array([outcome(q) for q in grid])
q_waste_ok = float(grid[max(int(np.argmax(stats[:, 0] > naive_waste)) - 1, 0)])   # waste <= today
q_met_ok = float(grid[int(np.argmax(stats[:, 1] >= naive_met))])                  # demand met >= today
q_out_ok = float(grid[int(np.argmax(stats[:, 2] <= naive_out))])                  # ran out <= today
lo, hi = max(q_met_ok, q_out_ok), q_waste_ok
print(f"\nBOUNDARIES vs today:  demand met >= today from q{q_met_ok:.3f}  |  "
      f"ran out <= today from q{q_out_ok:.3f}  |  waste <= today up to q{q_waste_ok:.3f}")
print(f"\nSTRICT-DOMINANCE WINDOW   q{lo:.3f} to q{hi:.3f}")
print("  Inside it the model beats the status quo on ALL THREE at once - less waste, more demand met,")
print("  fewer empty days - so it needs NO cost assumption to justify. Outside it you are making a")
print("  trade, and a trade requires a ratio you can defend.")
for q_star, label in ((lo, "window floor"), (hi, "window ceiling = waste-neutral")):
    w, m, k = outcome(q_star)
    print(f"    q{q_star:.3f} ({label:31s}) waste {w:8,.0f} vs {naive_waste:,.0f} | "
          f"met {m:.1f}% vs {naive_met:.1f}% | ran out {k:.1f}% vs {naive_out:.1f}%")

# The reader-facing map. Each comparison is its own column, stated as a direction, so nothing hides
# inside a summary verdict.
rows = []
for q_star in sorted({0.33, 0.40, round(lo, 3), 0.50, round(hi, 3), 0.60, 0.6667, 0.75, 0.80, 0.90, 0.95}):
    w, m, k = outcome(q_star)
    rows.append({"order at": f"q{q_star:.3f}", "implied c_u/c_o": round(q_star / (1 - q_star), 2),
                 "ran out %": round(k, 1), "empty days": "better" if k <= naive_out else "WORSE",
                 "demand met %": round(m, 1), "availability": "better" if m >= naive_met else "WORSE",
                 "waste % of demand": round(100 * w / total, 1),
                 "waste vs today %": round(100 * (1 - w / naive_waste), 0),
                 "waste": "better" if w <= naive_waste else "WORSE"})
qmap = pd.DataFrame(rows)
print("\n\nE5. THE ORDER PERCENTILE, TRANSLATED   validation, full-shelf days, recovered TFT")
print("    each row is self-consistent - the ratio shown is the one that q* implies, not a fixed one")
print(qmap.to_string(index=False))
print("\n    Read the three verdict columns together. Above the window the model buys availability")
print("    with waste. That is a legitimate choice - but it is a CHOICE, and it only becomes")
print("    defensible once the cost ratio behind it is.")

qmap.to_csv(REPORTS / "conclusion_quantile_map.csv", index=False)

TODAY (the naive rule): waste 10,792 units | demand met 79.9% | ran out on 41.7% of product-days



BOUNDARIES vs today:  demand met >= today from q0.361  |  ran out <= today from q0.461  |  waste <= today up to q0.513

STRICT-DOMINANCE WINDOW   q0.461 to q0.513
  Inside it the model beats the status quo on ALL THREE at once - less waste, more demand met,
  fewer empty days - so it needs NO cost assumption to justify. Outside it you are making a
  trade, and a trade requires a ratio you can defend.
    q0.461 (window floor                   ) waste    8,732 vs 10,792 | met 86.0% vs 79.9% | ran out 41.7% vs 41.7%
    q0.513 (window ceiling = waste-neutral ) waste   10,768 vs 10,792 | met 88.5% vs 79.9% | ran out 34.9% vs 41.7%




E5. THE ORDER PERCENTILE, TRANSLATED   validation, full-shelf days, recovered TFT
    each row is self-consistent - the ratio shown is the one that q* implies, not a fixed one
order at  implied c_u/c_o  ran out % empty days  demand met % availability  waste % of demand  waste vs today %  waste
  q0.330             0.49       60.0      WORSE          77.7        WORSE                9.6              56.0 better
  q0.400             0.67       50.0      WORSE          82.5       better               13.6              38.0 better
  q0.461             0.86       41.7     better          86.0       better               17.7              19.0 better
  q0.500             1.00       36.5     better          87.9       better               20.7               6.0 better
  q0.513             1.05       34.9     better          88.5       better               21.8               0.0 better
  q0.600             1.50       24.3     better          92.2       better               30.5             -3

In [13]:
anchors = {"today": None, "waste-focused\n(q0.461)": 0.461,
           "balanced\n(q0.513)": 0.513, "stockout-focused\n(q0.90)": 0.90}

# model_df, model_qcols, demand, mask, naive_order all come from E5 above - same validation,
# tft, recovered forecast, so this reuses rather than reloading it.
def score(order_arr, demand):
    sim = orders.simulate(order_arr, demand)
    return (100 * sim.stockout.mean(), 100 * (1 - sim.shortfall.sum() / demand.sum()),
            100 * sim.waste.sum() / demand.sum())

rows = []
for label, q in anchors.items():
    order_arr = naive_order if q is None else orders.order_quantity(model_df, q, model_qcols)[mask]
    so, dm, w = score(order_arr, demand)
    rows.append((label, so, dm, w))

fig, ax = plt.subplots(figsize=(12.5, 5.6), dpi=300)
labels = [r[0] for r in rows]
x = np.arange(len(labels))
metrics_ = {"stockout days": ("#2a78d6", [r[1] for r in rows]),
           "demand met": ("#1baf7a", [r[2] for r in rows]),
           "waste (% of demand)": ("#eb6834", [r[3] for r in rows])}
bar_w = 0.24
for i, (name, (color, vals)) in enumerate(metrics_.items()):
    offset = (i - 1) * (bar_w + 0.03)
    bars = ax.bar(x + offset, vals, width=bar_w, color=color, label=name, zorder=3)
    for xi, v in zip(x + offset, vals):
        ax.text(xi, v + 1.8, f"{v:.1f}%", ha="center", va="bottom", fontsize=12.5, color="#0b0b0b")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=14)
ax.set_ylim(0, 110)
ax.set_ylabel("% of demand / days", fontsize=13.5, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
ax.tick_params(axis="x", length=0)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_title("Three operating points against the status quo", fontsize=19, pad=16, loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False, fontsize=13,
         columnspacing=1.8)
fig.tight_layout()
fig.savefig(PLOTS / "quantile_anchors.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "quantile_anchors.png", POSTER_IMAGES / "quantile_anchors.png")
plt.close(fig)
print("saved outputs/plots/quantile_anchors.png (+ copied to poster/images/)")
print(pd.DataFrame(rows, columns=["anchor", "stockout_pct", "demand_met_pct", "waste_pct"]).to_string(index=False))


saved outputs/plots/quantile_anchors.png (+ copied to poster/images/)
                   anchor  stockout_pct  demand_met_pct  waste_pct
                    today     41.703130       79.906610  21.883869
  waste-focused\n(q0.461)     41.684377       85.951675  17.705616
       balanced\n(q0.513)     34.850190       88.531463  21.833463
stockout-focused\n(q0.90)      5.611118       97.975295  67.399295


### How precisely must the cost ratio be known?

A different frame from the table above, deliberately: that one kept every policy self-consistent
with its own implied ratio. Here **one** ratio is declared the truth and **every** policy is
scored under it — including the ones someone with a different belief would have chosen. That's
the only way to ask what being *wrong* about the ratio costs, which is the honest answer to "but
you don't know the ratio."

In [14]:
def cost_under(truth, q_star):
    o = orders.order_quantity(model_df, q_star, model_qcols)[mask]
    return float(truth * np.maximum(demand - o, 0).sum() + np.maximum(o - demand, 0).sum())

ratio_grid = np.arange(0.40, 0.96, 0.01)
rows = []
for truth in (1.0, 2.0, orders.HEADLINE_RATIO, 9.0):
    costs = np.array([cost_under(truth, q) for q in ratio_grid])
    near = ratio_grid[costs <= 1.05 * costs.min()]
    rows.append({"if the truth is c_u/c_o": truth,
                 "newsvendor says": round(truth / (truth + 1), 3),
                 "cheapest actually at": round(float(ratio_grid[costs.argmin()]), 2),
                 "within 5% of cheapest": f"q{near.min():.2f}-q{near.max():.2f}",
                 "penalty for using q0.80 %": round(100 * (cost_under(truth, 0.80) / costs.min() - 1), 1)})
ratio_sensitivity = pd.DataFrame(rows)
print("E6. ONE TRUTH AT A TIME, EVERY POLICY SCORED UNDER IT")
print(ratio_sensitivity.to_string(index=False))
print("\n    Two readings, and the second is a finding rather than a reassurance:")
print("    1. The cost curve is FLAT near its floor - a band of q* roughly 0.2 wide sits within 5% of")
print("       the cheapest. So the ratio only has to be roughly right. Being in the wrong")
print("       neighbourhood is expensive; being imprecise is not. That is what makes an unknown")
print("       ratio survivable, and it is why the sweep is a sensitivity analysis rather than a gap.")
print("    2. The cheapest policy is BELOW what the newsvendor rule predicts, at every truth tested.")
print("       The rule is not wrong - it is being handed a forecast that runs ~8.5% high (see the")
print("       accuracy table in §7, WPE +0.085). Conformal corrects band WIDTH, never CENTRING,")
print("       so nothing in this pipeline removes that median bias, and reading the q*-th percentile")
print("       off it over-orders. The fix belongs upstream in the forecast; shading q* down here")
print("       would be tuning on the window the result is reported on.")

ratio_sensitivity.to_csv(REPORTS / "conclusion_ratio_sensitivity.csv", index=False)

E6. ONE TRUTH AT A TIME, EVERY POLICY SCORED UNDER IT
 if the truth is c_u/c_o  newsvendor says  cheapest actually at within 5% of cheapest  penalty for using q0.80 %
                     1.0            0.500                  0.40           q0.40-q0.49                       84.5
                     2.0            0.667                  0.53           q0.44-q0.62                       35.4
                     4.0            0.800                  0.65           q0.56-q0.75                        9.9
                     9.0            0.900                  0.80           q0.70-q0.90                        0.0

    Two readings, and the second is a finding rather than a reassurance:
    1. The cost curve is FLAT near its floor - a band of q* roughly 0.2 wide sits within 5% of
       the cheapest. So the ratio only has to be roughly right. Being in the wrong
       neighbourhood is expensive; being imprecise is not. That is what makes an unknown
       ratio survivable, and it is why t

## 6b. New check — do the validation-chosen quantiles survive the test week?

The three anchors above (`waste-focused`, `balanced`, `stockout-focused`) are quantiles picked on
**validation**. This section takes those exact, already-fixed quantiles and applies them, unchanged,
to the **sealed test week** — for both model families — to see whether the choice made on
validation actually holds up out of sample. This had never been run before; it exists because the
poster claims the three anchors as safe choices, and that claim should survive contact with the
test week the same way every other number here is required to.

In [15]:
ANCHOR_Q = {"waste_focused": 0.461, "balanced": 0.513, "stockout_focused": 0.90}

rows = []
for family in ("tft", "xgb"):
    for period in ("validation", "test"):
        df, qcols = orders.load_forecast(period=period, family=family, target="recovered")
        demand, mask = orders.realised_demand(df, regime="observed")
        naive_order = orders.naive_orders(df)

        so, dm, w = score(naive_order[mask], demand)
        rows.append(dict(family=family, period=period, anchor="naive_today",
                         stockout_pct=round(so, 1), demand_met_pct=round(dm, 1), waste_pct=round(w, 1)))
        for name, q in ANCHOR_Q.items():
            order_arr = orders.order_quantity(df, q, qcols)[mask]
            so, dm, w = score(order_arr, demand)
            rows.append(dict(family=family, period=period, anchor=name,
                             stockout_pct=round(so, 1), demand_met_pct=round(dm, 1), waste_pct=round(w, 1)))

transfer = pd.DataFrame(rows)
transfer.to_csv(REPORTS / "quantile_transfer_to_test.csv", index=False)
print(transfer.to_string(index=False))
print()
print("Reading: waste-focused and balanced were chosen on validation because they matched or beat")
print("naive on stockouts there. On the test week, both arms show BOTH anchors doing WORSE than")
print("naive on stockouts and demand met (only cheaper on waste) - only stockout-focused (q0.90)")
print("stays unambiguously better than naive on the test week, in both model families.")


family     period           anchor  stockout_pct  demand_met_pct  waste_pct
   tft validation      naive_today          41.7            79.9       21.9
   tft validation    waste_focused          41.7            86.0       17.7
   tft validation         balanced          34.9            88.5       21.8
   tft validation stockout_focused           5.6            98.0       67.4
   tft       test      naive_today          42.8            80.5       18.6
   tft       test    waste_focused          60.6            73.0        7.8
   tft       test         balanced          53.9            76.3       10.1
   tft       test stockout_focused          16.0            92.9       39.5
   xgb validation      naive_today          41.7            79.9       21.9
   xgb validation    waste_focused          40.0            86.0       19.0
   xgb validation         balanced          32.9            88.8       23.6
   xgb validation stockout_focused           4.5            98.3       74.4
   xgb      

## 7. Why TFT and recovered data

### Forecast accuracy, scored on non-stockout (clean) rows

Computed here from the four validation forecast parquets, not read from
`forecast_vs_baselines.csv` — the latter is an older, TFT-only scorecard from before the
XGBoost arms were added, and would silently drop `xgb_recovered`/`xgb_raw` from the comparison
the poster's Card 7 actually makes.

In [16]:
val_fc = {f"{f}_{t}": pd.read_parquet(config.forecast_parquet("validation", t, f)) for f, t in ARMS}
arms_acc = pd.DataFrame({n: {**quantile_scores(d), "pinball@0.8": pinball_by_quantile(d)["pinball@0.8"]}
                         for n, d in val_fc.items()}).T
accuracy = pd.concat([arms_acc,
                      pd.read_csv(config.BASELINE_SCORECARD, index_col=0)[["WAPE", "WPE"]]]).round(4)
accuracy.to_csv(REPORTS / "conclusion_accuracy.csv")
print("B. FORECAST ACCURACY  validation, non-stockout rows, against recorded sale_amount")
print(accuracy.to_string())
print()
print("pinball@0.8 is the only accuracy column that converts into money - it is the quantile the")
print("order is read at. Recovered beats raw there in BOTH families, while pooled WAPE ties.")
print()
print(f"TFT vs XGBoost, recovered: pinball {accuracy.loc['tft_recovered', 'pinball(avg)']:.4f} "
      f"vs {accuracy.loc['xgb_recovered', 'pinball(avg)']:.4f} "
      f"({100*(1 - accuracy.loc['tft_recovered','pinball(avg)']/accuracy.loc['xgb_recovered','pinball(avg)']):.1f}% better)")

B. FORECAST ACCURACY  validation, non-stockout rows, against recorded sale_amount
                    WAPE     WPE     MAE  pinball(avg)   CRPS~  pinball@0.8
tft_recovered     0.3284  0.0851  0.3358        0.1074  0.2091       0.1281
tft_raw           0.3286 -0.0656  0.3364        0.1089  0.2120       0.1300
xgb_recovered     0.3417  0.1005  0.3502        0.1116  0.2173       0.1330
xgb_raw           0.3416 -0.0218  0.3495        0.1118  0.2176       0.1362
seasonal_naive    0.4213  0.0187     NaN           NaN     NaN          NaN
xgboost_quantile  0.3423 -0.0187     NaN           NaN     NaN          NaN
sarima            0.5210 -0.2521     NaN           NaN     NaN          NaN

pinball@0.8 is the only accuracy column that converts into money - it is the quantile the
order is read at. Recovered beats raw there in BOTH families, while pooled WAPE ties.

TFT vs XGBoost, recovered: pinball 0.1074 vs 0.1116 (3.8% better)


### The payoff, quantified — lost sales recovered vs. waste added

At q50, on full-shelf days only (the regime that *penalises* recovery), for both model families.
`worth_it_above_ratio` = waste added ÷ lost sales recovered: how much worse an empty shelf has to be
than a bin before recovery pays. Below 1.0 means it pays even if the two cost exactly the same.

In [17]:
band = censoring_bucket(daily)

for family in ("tft", "xgb"):
    raw_df, raw_qcols = orders.load_forecast(period="validation", family=family, target="raw")
    rec_df, rec_qcols = orders.load_forecast(period="validation", family=family, target="recovered")
    trade = lost_sales_vs_waste({"raw": raw_df, "recovered": rec_df}, band)
    print(f"--- {family} ---")
    print(trade.to_string())
    print()


--- tft ---
               lost_sales_%_raw  waste_%_raw  lost_sales_%_recovered  waste_%_recovered  lost_sales_recovered_pts  waste_added_pts  worth_it_above_ratio
band                                                                                                                                                    
<25% censored              22.8         19.5                    17.6               24.4                       5.2              4.9                  0.94
25-50%                     19.8         15.4                    12.8               23.3                       7.0              7.9                  1.13
50-75%                     18.1         11.6                    10.8               19.0                       7.3              7.4                  1.01
>=75%                      22.7          2.8                     9.8                9.1                      12.9              6.3                  0.49



--- xgb ---
               lost_sales_%_raw  waste_%_raw  lost_sales_%_recovered  waste_%_recovered  lost_sales_recovered_pts  waste_added_pts  worth_it_above_ratio
band                                                                                                                                                    
<25% censored              22.7         24.0                    17.4               28.6                       5.3              4.6                  0.87
25-50%                     17.6         18.2                    12.8               24.1                       4.8              5.9                  1.23
50-75%                     16.1         14.6                    10.0               21.2                       6.1              6.6                  1.08
>=75%                      25.0          4.2                    11.3               11.1                      13.7              6.9                  0.50



## 8. Limitations

No computation here — see `poster/poster.html` Card 8 and README §7 for the quick list, and
the fuller limitations table in the Conclusion below. The one number worth re-stating: the
chronic-stockout headline (§4 above) rests on **1,375** scored rows — the smallest band in the
per-censoring-bucket table.

## 9. Conclusion

`sale_amount` is normalised by an undisclosed coefficient, so every figure in this notebook is a
**ratio or a percentage** — no absolute quantity is claimed anywhere in this project.

---

### How to read the numbers

Six terms carry most of the weight. All six are easy to misread.

**`q*` — the order percentile.** How far up the predicted demand distribution we aim. It comes from
the two costs and nothing else:

$$q^* = \frac{c_u}{c_u + c_o}$$

where `c_u` is the cost of a lost sale and `c_o` the cost of a binned unit. If a stockout hurts 4×
as much as a bin, `q* = 4/(4+1) = 0.80` — order the 80th percentile of predicted demand. Equal
costs → order the median. The real cost ratio isn't in the data, so it is **swept across nine
values** rather than guessed, and only conclusions that survive the whole sweep are claimed.

**The naive rule — the status quo we have to beat.** *"Order exactly what sold on this same weekday
last week"*: raw recorded sales at `dt − 7 days`, read from the un-recovered history a real store
would actually have. No model, no calibration. Unglamorous on purpose — a percentage improvement is
meaningless without a real baseline.

**`stockout %` vs `demand met %` — not the same thing, and the difference matters.** Stockout % counts
**days**: run out by a hair at 9pm and the whole day counts as a stockout. Demand met % counts
**units**: running out by a hair barely registers. So "9% of days stocked out but 97% of demand met"
is not a contradiction — it's the same policy described two ways. Stockout % is the operational
headache; demand met % is the revenue.

**The `95% CI` on coverage — a day-block bootstrap.** Take the dates in the window, resample whole
**dates** with replacement 2,000 times, recompute coverage each time, and report the 2.5th–97.5th
percentiles. It answers: *if we'd observed a different but comparable fortnight, where would coverage
plausibly have landed?* **If the nominal level (0.80) sits inside the interval, the band is
statistically indistinguishable from correct; if it sits outside, the miss is real.** Whole dates
rather than individual rows because 5,601 products share every date — one busy Saturday moves
thousands of rows together, and the measured day-to-day spread is **13.7×** what independence would
predict. That is why the textbook test (Kupiec) was dropped: it assumes independent rows, so it is
handed roughly fourteen times more evidence than exists and rejects a two-point miss.

**The `+0.03` drift inflation — ask for 83% to get 80%.** The calibration offset is fitted on one
fortnight and then applied to a *later* one, which breaks the exchangeability that conformal
prediction's guarantee rests on. Coverage decays with distance in time — measured at roughly **2.5
points per fortnight**. So for any window that sits *forward* of the calibration set, the code asks
for `0.80 + 0.03 = 0.83` coverage in order to land near 0.80 when it gets there. Mechanically it does
nothing exotic: it reads a **higher percentile of the same list of past errors**, so the offset moves
from 0.0775 to 0.1132. It was measured by fitting on validation and applying to calibration — the
same one-window-forward jump — so **the test week was never used to choose it**, only to check it.
`conformal._is_forward` reads the frozen calendar and switches it on for the test week and nothing
else.

**`orders at q0.79` vs `q0.68` — the one that genuinely looks wrong.** These are percentiles of
**different distributions**, so they are not comparable as order sizes. Each arm has its own
forecast, and the recovered arm's whole distribution sits **higher** than the raw arm's — so
"the 68th percentile of the recovered forecast" and "the 79th percentile of the raw forecast" can put
*the same number of units* on the shelf.

What the number actually reports is **how far into its own upper tail each model has to reach to keep
shelves full**:

- The **raw** model learned from sales that stockouts had already truncated, so its distribution sits
  low. To get enough units out of it, you must reach the **79th** percentile.
- The **recovered** model learned from filled-in demand, so it is already centred near the truth. The
  **68th** percentile is enough.

And that is why waste differs at *identical* availability. Reaching further into the tail is a blunt
instrument: it adds units in proportion to each row's **spread**, not to each row's **demand**, so it
piles stock onto uncertain days whether or not those days are busy. The recovered model doesn't need
that padding, so at the same availability its units land closer to where demand actually is and less
of it ends up in the bin. **A lower required percentile is a statement about calibration, not about
ordering less.**

**"~23% vs ~4%" — where those two numbers come from, and why to state them carefully.** The 23% is the
WAPE improvement on chronic-stockout products from changing the **training target** (raw → recovered):
−25.2% for the TFT, −22.9% for XGBoost. The 4% is the **architecture** gap, TFT vs XGBoost on pooled
pinball (0.1074 vs 0.1116 = 3.9%). **These are different metrics on different row sets** — a WAPE
change on 1,375 rows against a pinball change on 47,994. So the comparison is indicative of where the
effort paid off, *not* a like-for-like ratio, and it is better said as two separate sentences than as
one "23% vs 4%" headline.

## What the tables say

The numbers are printed above. This is the reading.

### The six claims, and where each one is anchored

| # | Claim | Evidence | Window | Could it have failed? |
|---|---|---|---|---|
| 1 | Censored demand is recoverable | beats a no-model control by **19.5%** on days where recorded sales *are* demand | held-out full-shelf days | **Yes** |
| 2 | Recovery fixes the forecast where censoring bites | **−25.2%** (TFT) / **−22.9%** (XGB) WAPE on ≥75%-censored series; bias −0.20 → +0.00 | validation | **Yes** — it does *not* help in the other three bands |
| 3 | Not an artefact of one architecture | both families agree, on **both** windows | validation **+ test** | **Yes** — they could have disagreed |
| 4 | Intervals can be made honest post-hoc | uncorrected test coverage 0.60–0.86 → corrected 0.76–0.96 | validation **+ test** | **Yes** — and it only partly succeeds, see below |
| 5 | **At equal availability, recovery wastes less** | **−2.6 / −5.3 pts** on validation; **−20.8 / −20.1 pts** on test | validation **+ test, both families** | **Yes** — the sign could have flipped |
| 6 | Ordering beats the status quo | cheaper at **all nine** cost ratios, and for every arm on both windows | validation **+ test** | **Yes** — one losing ratio breaks it |

**The single sentence:** *recovering the demand that stockouts hid barely changes forecast accuracy and
still produces a materially better order — because it removes a directional bias that accuracy metrics
charge nothing for.*

---

### The headline result, now complete

**Claim 5 replicates on the sealed test week, in both model families, and four times larger than on
validation** (E2):

| window | model | raw wastes | recovered wastes | saved | raw must order at | recovered needs |
|---|---|---|---|---|---|---|
| validation | TFT | 44.0% | 41.4% | −2.6 pts | q0.80 | q0.70 |
| validation | XGBoost | 48.5% | 43.2% | −5.3 pts | q0.79 | q0.68 |
| **test** | **TFT** | **72.7%** | **51.9%** | **−20.8 pts** | q0.95 | q0.92 |
| **test** | **XGBoost** | **95.9%** | **75.8%** | **−20.1 pts** | q0.97 | q0.94 |

Same availability, four arms, two windows, one direction. The mechanism is the `orders at` pair: the
raw models must be driven further into their own upper tail to keep shelves full, and that padding is
what fills bins.

**And on the test week the confound disappears entirely** (E3). At the fixed headline `q*=0.80`,
*validation* makes the raw arms look cheaper — they order less, so they waste less, and they pay for
it in availability. On test, recovered beats raw on **every column**:

| window | arm | cost vs naive | ran out | demand met |
|---|---|---|---|---|
| test | **tft_recovered** | **+28.0%** | 22.6% | 90.2% |
| test | tft_raw | +9.2% | 29.2% | 84.6% |
| test | **xgb_recovered** | **+13.4%** | 19.4% | 89.2% |
| test | xgb_raw | **+0.8%** | 26.7% | 84.1% |

**That `+0.8%` is the business case in one number.** A model trained on censored sales — conformally
calibrated and ordered by an optimal newsvendor rule — beats *"order what sold last week"* by **under
one percent** on the sealed week. Its recovered twin beats the same baseline by 13.4%, and the
recovered TFT by 28.0%.

---

### Three things that are less comfortable

**Calibration only partly works out of sample.** In D, **0 of 8** validation combinations contain their
nominal level (all slightly *over*-wide) and **5 of 8** on test — but read which: all four 95% bands
contain nominal, while only **one of four 80% bands** does (`tft_recovered`). The rest undershoot at
0.76–0.77. Uncorrected test coverage runs 0.60–0.86, so the drift is real and one fixed `+0.03` does
not fully absorb it at 80%. Note too that `+0.03` was measured on the **80%** band and applied
unchanged to the 95% band, where asking for 98% is a far more aggressive move; it lands, but it was
never tuned for that.

**Everything degrades with distance from the calibration window.** The recovered TFT meets 96.9% of
demand on validation and **90.2%** on test at the same `q*`; its cost edge falls 34.7% → 28.0%. **Any
validation figure is the optimistic one.** A production system would refit the conformal offset on a
rolling basis rather than once.

**The headline `q*` is not the cost-minimising one, and the gap has a clean interpretation** (E6).
Score every policy under one declared truth at a time and the cheapest sits consistently *below* what
the newsvendor rule predicts:

| if the truth is | rule says order at | actually cheapest at | cost of using q0.80 |
|---|---|---|---|
| 1:1 | q0.50 | **q0.40** | +84.5% |
| 2:1 | q0.67 | **q0.53** | +35.4% |
| **4:1** *(the headline)* | q0.80 | **q0.65** | **+9.9%** |
| 9:1 | q0.90 | **q0.80** | +0.0% |

Read the last row: **ordering at q0.80 is the optimal policy for a 9:1 world, not a 4:1 one.** So the
project states a 4:1 assumption while *implementing* something closer to 9:1 — it is buying more
availability than its own stated cost ratio justifies, at about **10% excess cost**.

That is not the newsvendor rule failing. It is the rule being handed a forecast that runs ~8.5% high
(§7, `WPE +0.085`). **Conformal corrects band *width*, never *centring*,** so nothing in this pipeline
removes the recovered forecast's median bias, and reading the `q*`-th percentile off a distribution
shifted high over-orders by roughly 0.15 of a quantile. The fix belongs upstream in the forecast;
shading `q*` down here would be tuning on the window the result is reported on.

---

### Choosing an operating point, without assuming a cost ratio

`q*` and `c_u/c_o` are the same number twice, and neither means anything to a retailer. E5 translates
both into outcomes and compares each against **today**, on three axes separately — waste, demand met,
and empty-shelf days. It deliberately does **not** collapse them into a single verdict: a tolerable
waste level is a business constraint nobody supplied, and inventing one is exactly what the cost sweep
exists to avoid.

Doing that exposes a band worth naming — **the strict-dominance window, q0.46 to q0.51**, inside which
the model beats the status quo on *all three* at once:

| | today | q0.461 (floor) | q0.513 (ceiling) |
|---|---|---|---|
| waste | 10,792 | **8,732** *(−19%)* | 10,768 *(level)* |
| demand met | 79.9% | **86.0%** | **88.5%** |
| ran out on | 41.7% of days | 41.7% *(level)* | **34.9%** |

**This is the project's one assumption-free recommendation.** At q0.46 it wastes **19% less than today
at identical stockout frequency**, and still meets 6 more points of demand. At q0.51 it holds waste
level and cuts empty days by 7 points. Neither requires a cost ratio — they are *constraints*
("no worse than today"), not trade-offs, so there is nothing to defend.

**Outside the window every operating point is a trade**, and a trade needs a ratio you can argue for.
Above q0.51 the model buys availability with waste — legitimate, but a choice: at q0.95 it meets 99%
of demand while wasting **4.2× today's**. Below q0.46 it empties shelves more often than the status
quo, which rules out the entire waste-minimising end of the sweep as a real option.

---

## Limitations

| | Effect |
|---|---|
| **Demand during a stockout is unobservable** | the load-bearing one. Validation happens on full-shelf days; recovery is only ever *applied* to days the shelf emptied. **No test can close that gap** |
| `recovered_demand` is a floor, not a measurement | a model output floored at recorded sales — demonstrably closer than raw sales, and if it errs it errs short |
| **Nothing corrects the recovered forecast's centring** | it runs ~8.5% high and conformal only widens, so the newsvendor `q*` over-orders by ~0.15 of a quantile: q0.80 implements a 9:1 belief while 4:1 is stated, at ~10% excess cost (E6) |
| The headline recovery band is **1,375 rows** | the smallest cell in the table, from a single seeded subset draw (11.2% of the corpus) |
| Recovery's validation is a worst case | held-out days are blanked across all 16 active hours, harsher than most real censored days; the typical-day error is unmeasured. The bound errs *against* the model |
| Stage-1 history features are themselves censored | `lag7`/`roll7` are built from recorded sales, biasing recovery **downward** — the same direction as the floor |
| The raw TFT's tuning table was never saved | its encoder length is borrowed from the recovered arm and verified against its saved validation forecast (notebook 04, §4) — sound, but a reconstruction rather than a record |
| Test-week coverage intervals rest on **7 days** | a day-block bootstrap over 7 blocks is coarse; "contains nominal" is a weaker statement there than on validation's 14 |
| One demand regime scores against `recovered_demand` | not a neutral referee; both regimes are reported as a **bound** rather than one being chosen |
| The cost ratio is an assumption | swept across nine values; only sweep-wide claims are made |
| Waste is arithmetic, never observed | spoilage is not recorded in the dataset |
| The newsvendor is single-period | shelf life is unknown, so leftovers cannot carry forward |
| Weather enters as **realised**, not forecast | flatters absolute accuracy; baselines get the same covariates, so relative comparisons hold |
| The TFT's ~4% accuracy edge comes from a comparison that favours it | it early-stops on the window it is then scored on; the tree stops on held-back *training* days |
| SARIMA is fitted on 30 sampled series | a reference point, not a like-for-like competitor |
| The ordering results carry **no error bars** | calibration is the only layer with uncertainty quantification; the waste and cost gaps are point estimates |
| Units are normalised | no saving can be quoted in currency |

---

## What this project demonstrates — and what it does not

**Demonstrates:**

1. **Censored demand is recoverable from stockout annotations**, and the recovery survives a test that
   could have falsified it.
2. **The benefit lands exactly where the mechanism predicts** — chronic-stockout products — and is
   invisible pooled, because pooled scoring can only look at the days recovery does not change.
3. **Accuracy was never the mechanism.** Recovery buys ≈0% pooled WAPE and still produces a better
   order. `pinball@0.8` — the only accuracy column that converts into money, being the quantile the
   order is read at — favours recovered in *both* families while WAPE ties.
4. **Prediction intervals can be calibrated after the fact**, and their failure mode under time drift
   is measurable and partly correctable *without* consulting the sealed window.
5. **At matched availability, recovery wastes less** — in both families, on both windows, under the
   regime biased against it, and four times more strongly out of sample than in.
6. **The data layer mattered more than the model.** Changing the training target moved the
   chronic-stockout group by ~23%; changing architecture moved pooled pinball by ~4%. Different
   metrics on different row sets, so not a like-for-like ratio — but the direction is not in doubt,
   and testing it is exactly what the four-arm design was built for.

**Does not demonstrate:** that any of this survives contact with real inventory data. This is a proof
of concept on a dataset that records **sales, not stock**. Stock on hand and deliveries would turn
recovery's central assumption from an argument into a measurement; recorded spoilage would make waste
an observed outcome rather than an arithmetic one; shelf life would make the newsvendor multi-period;
real unit costs would collapse the nine-ratio sweep to one operating point with a figure in currency.
**None of them require the architecture to change** — each replaces an assumption with a measurement
at a point the design already isolates.

## Summary — headline numbers this notebook reproduces

| poster claim | value | source in this notebook |
|---|---|---|
| Recovery WAPE vs. naive | 0.2966 vs. 0.3684 (-19.5%) | §4 |
| Chronic-stockout WAPE change | -25.2% (TFT) / -22.9% (XGBoost) | §4 |
| 80%/95% band coverage, both windows, all 4 arms | see table | §4 |
| Waste at 95% demand met, both windows | see `waste_comparison.png` | §4 |
| Cheaper than naive at all 9 cost ratios | 20.5%–72.6% | §5 |
| Three operating-point anchors | see `quantile_anchors.png` | §6 |
| How precisely must the cost ratio be known? | flat within ±0.1 of q*, cheapest sits below the newsvendor prediction | §6 |
| **New**: do those anchors hold on the test week? | see `quantile_transfer_to_test.csv` | §6b |
| TFT vs. XGBoost pinball | 0.1074 vs. 0.1116 | §7 |
| Recovery pays once a stockout costs more than ~0.5×–0.87× a bin | see trade tables | §7 |
| Full six-claim synthesis and limitations | — | §9 |